In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Load the Master Dataset and Build The Month Spine From It**

In [10]:
merged = pd.read_csv("Assistance_Demographics_Master.csv", dtype={"Zip Code": str})
for flag in ["Calfresh_Outlier", "LIHEAP_Outlier", "FoodBank_Outlier"]:
    merged[flag] = merged[flag].astype("boolean")

In [11]:
months = pd.PeriodIndex(merged["Month Year"], freq="M").unique().sort_values()
full = pd.period_range(months.min(), months.max(), freq="M")
assert full.difference(months).empty, "master dataset has missing months"
print(f"Months: {months.min()} to {months.max()} ({len(months)})")

Months: 2018-01 to 2025-08 (92)


**Define the columns and events associated with policy**

In [12]:
cols = ["policy", "program", "change_type", "start_month", "end_month",
        "expected_direction", "rationale"]

In [13]:
events = pd.DataFrame([
    # ---------------- CalFresh / SNAP ----------------
    ("2018 Farm Bill", "CalFresh", "eligibility", "2018-12", "2019-02", "uncertain",
     "Kept ABAWD limits; cut state exemption share 15% to 12%"),
    ("SSI cash-out reversal", "CalFresh", "eligibility", "2019-06", "2019-08", "increase",
     "About 500,000 SSI recipients newly eligible statewide"),
    ("Emergency Allotments", "CalFresh", "emergency_provision", "2020-03", "2023-03", "increase",
     "Every household raised to maximum benefit; makes enrolling more worthwhile"),
    ("Emergency Allotments", "FoodBank", "emergency_provision", "2020-03", "2023-03", "decrease",
     "Larger food benefits reduce need for food bank support"),
    ("Pandemic SNAP waivers (ABAWD limit paused, student exemptions)", "CalFresh",
     "emergency_provision", "2020-03", "2023-06", "increase",
     "Work-rule time limit paused; more students eligible"),
    ("Temporary 15% SNAP increase", "CalFresh", "emergency_provision", "2021-01", "2021-09", "increase",
     "Higher maximum benefit"),
    ("Temporary 15% SNAP increase", "FoodBank", "emergency_provision", "2021-01", "2021-09", "decrease",
     "Higher food benefits"),
    ("Thrifty Food Plan update (+21%)", "CalFresh", "benefit_level", "2021-10", "2021-12", "increase",
     "Permanent benefit increase"),
    ("Thrifty Food Plan update (+21%)", "FoodBank", "benefit_level", "2021-10", "2021-12", "decrease",
     "Permanent benefit increase"),
    ("Emergency Allotments end", "CalFresh", "benefit_level", "2023-04", "2023-06", "uncertain",
     "Benefits fall about $163/household but eligibility unchanged"),
    ("Emergency Allotments end", "FoodBank", "benefit_level", "2023-04", "2023-06", "increase",
     "Households replace lost benefits with food bank support"),
    ("ABAWD time limit resumes; student exemptions end", "CalFresh", "eligibility", "2023-07", "2023-09",
     "decrease", "Work rules return after the Public Health Emergency"),
    ("ABAWD time limit resumes; student exemptions end", "FoodBank", "eligibility", "2023-07", "2023-09",
     "increase", "People losing CalFresh turn to food banks"),
    ("Fiscal Responsibility Act ABAWD changes", "CalFresh", "eligibility", "2023-09", "2024-10", "uncertain",
     "Age limit rises 49 to 54 (restriction) but new exemptions added (expansion)"),
    ("One Big Beautiful Bill Act", "CalFresh", "eligibility", "2025-07", "2025-09", "decrease",
     "Work rules extended to ages 18-64; exemptions removed"),
    ("One Big Beautiful Bill Act", "FoodBank", "eligibility", "2025-07", "2025-09", "increase",
     "People losing CalFresh turn to food banks"),
    # ---------------- LIHEAP ----------------
    ("CARES Act LIHEAP supplement ($900M)", "LIHEAP", "funding", "2020-04", "2020-09", "increase",
     "FY2020 supplemental funding"),
    ("CPUC utility shutoff moratorium", "LIHEAP", "emergency_provision", "2020-03", "2021-09", "decrease",
     "No shutoff threat reduces urgent applications"),
    ("ARPA LIHEAP supplement ($4.5B)", "LIHEAP", "funding", "2021-03", "2022-09", "increase",
     "Supplemental funding available through FY2022"),
    ("Shutoff moratorium ends; payment plans", "LIHEAP", "emergency_provision", "2021-10", "2021-12", "increase",
     "Shutoffs possible again for households with arrears"),
    ("IIJA LIHEAP funding (+$100M/yr)", "LIHEAP", "funding", "2021-11", "2022-01", "increase",
     "Added annual funding FY2022-FY2026"),
    ("FY2023 LIHEAP funding ($6.1B)", "LIHEAP", "funding", "2022-10", "2023-09", "increase",
     "Highest regular-year funding, incl. $2B supplemental"),
    ("CAPP round 2 utility debt credits", "LIHEAP", "funding", "2022-11", "2023-01", "decrease",
     "Arrears cleared, reducing need for bill help"),
    ("FY2024 LIHEAP funding cut to $4.125B", "LIHEAP", "funding", "2023-10", "2023-12", "decrease",
     "Less funding for assistance"),
    ("Federal LIHEAP staff dismissed; FY2025 funds held", "LIHEAP", "funding", "2025-04", "2025-05", "decrease",
     "Payments to states delayed until May 1, 2025"),
], columns=cols)

VALID_TYPES = {"eligibility", "benefit_level", "emergency_provision", "funding"}
VALID_DIRS = {"increase", "decrease", "uncertain"}
assert set(events["change_type"]) <= VALID_TYPES
assert set(events["expected_direction"]) <= VALID_DIRS

**Expand Each Event to One Row Per Month, Clipped To The Data Range**

In [14]:
rows = []
for e in events.itertuples(index=False):
    for m in pd.period_range(e.start_month, e.end_month, freq="M"):
        if months.min() <= m <= months.max():
            rows.append({**e._asdict(), "Month Year": str(m)})
long = pd.DataFrame(rows)

**Collaspe to One Row Per Month with Columns Per Program**

In [15]:
def combine_directions(dirs):
    """All policies agree -> that direction; any conflict -> uncertain."""
    unique = set(dirs)
    return unique.pop() if len(unique) == 1 else "uncertain"

annotations = pd.DataFrame({"Month Year": months.astype(str)})
for prog in ["CalFresh", "LIHEAP", "FoodBank"]:
    g = long[long["program"] == prog]
    agg = (g.groupby("Month Year")
             .agg(types=("change_type", lambda x: "; ".join(sorted(set(x)))),
                  direction=("expected_direction", combine_directions),
                  policies=("policy", lambda x: "; ".join(dict.fromkeys(x))))
             .reset_index()
             .rename(columns={"types": f"{prog}_Change_Types",
                              "direction": f"{prog}_Expected_Direction",
                              "policies": f"{prog}_Policies"}))
    annotations = annotations.merge(agg, on="Month Year", how="left")
    active = annotations[f"{prog}_Change_Types"].notna()
    annotations.insert(annotations.columns.get_loc(f"{prog}_Change_Types"),
                       f"{prog}_Policy_Active", active.map({True: "Yes", False: "No"}))

# Which programs have data that month (from the master dataset)
coverage = merged.groupby("Month Year").agg(
    CalFresh_Data_Available=("Calfresh_Dollars", lambda s: s.notna().any()),
    LIHEAP_Data_Available=("LIHEAP_Enrollment_Count", lambda s: s.notna().any()),
    FoodBank_Data_Available=("Total_Pounds_Distributed", lambda s: s.notna().any()),
).replace({True: "Yes", False: "No"}).reset_index()
annotations = annotations.merge(coverage, on="Month Year", how="left")
annotations["Any_Policy_Active"] = (
    annotations[[f"{p}_Policy_Active" for p in ["CalFresh", "LIHEAP", "FoodBank"]]] == "Yes"
).any(axis=1).map({True: "Yes", False: "No"})

**Validate the Data**

In [16]:
assert len(annotations) == len(months)
assert annotations["Month Year"].is_unique
apr23 = annotations.set_index("Month Year").loc["2023-04"]
assert apr23["FoodBank_Expected_Direction"] == "increase"
assert "benefit_level" in apr23["CalFresh_Change_Types"]
assert annotations.set_index("Month Year").loc["2023-07", "CalFresh_Expected_Direction"] == "decrease"
for c in [c for c in annotations.columns if c.endswith("_Policy_Active")]:
    print(c, annotations[c].value_counts().to_dict())

CalFresh_Policy_Active {'Yes': 64, 'No': 28}
LIHEAP_Policy_Active {'Yes': 48, 'No': 44}
FoodBank_Policy_Active {'No': 47, 'Yes': 45}
Any_Policy_Active {'Yes': 66, 'No': 26}


**Export the Data**

In [17]:
annotations.to_csv("Policy_Annotations.csv", index=False)
events.to_csv("Policy_Events.csv", index=False)  # documents every flag's source and reasoning
print("Saved Policy_Annotations.csv and Policy_Events.csv")

Saved Policy_Annotations.csv and Policy_Events.csv


In [18]:
annotated_master = merged.merge(annotations, on="Month Year", how="left")

In [19]:
annotated_master

,Zip Code,Month Year,Calfresh_Dollars,Calfresh_Meal_Cost,Calfresh_Meals,Calfresh_Outlier,LIHEAP_Enrollment_Count,LIHEAP_Outlier,FSD_Pounds,SDFB_Pounds,...,LIHEAP_Expected_Direction,LIHEAP_Policies,FoodBank_Policy_Active,FoodBank_Change_Types,FoodBank_Expected_Direction,FoodBank_Policies,CalFresh_Data_Available,LIHEAP_Data_Available,FoodBank_Data_Available,Any_Policy_Active
0,91901,2018-01,NaN,NaN,NaN,<NA>,NaN,<NA>,7884.0,0.0,...,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
1,91901,2018-02,NaN,NaN,NaN,<NA>,NaN,<NA>,10558.0,0.0,...,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
2,91901,2018-03,NaN,NaN,NaN,<NA>,NaN,<NA>,10642.0,0.0,...,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
3,91901,2018-04,NaN,NaN,NaN,<NA>,NaN,<NA>,12192.0,0.0,...,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
4,91901,2018-05,NaN,NaN,NaN,<NA>,NaN,<NA>,10370.0,0.0,...,NaN,NaN,No,NaN,NaN,NaN,No,No,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13591,92198,2025-04,7966.02,4.625089,1722.349671,False,NaN,<NA>,NaN,NaN,...,decrease,Federal LIHEAP staff dismissed; FY2025 funds held,No,NaN,NaN,NaN,Yes,No,Yes,Yes
13592,92198,2025-05,9157.02,4.653009,1967.978089,False,NaN,<NA>,NaN,NaN,...,decrease,Federal LIHEAP staff dismissed; FY2025 funds held,No,NaN,NaN,NaN,Yes,No,Yes,Yes
13593,92198,2025-06,10087.03,4.631940,2177.711556,False,NaN,<NA>,NaN,NaN,...,NaN,NaN,No,NaN,NaN,NaN,Yes,No,Yes,No
13594,92198,2025-07,9229.00,4.725154,1953.163959,False,NaN,<NA>,NaN,NaN,...,NaN,NaN,Yes,eligibility,increase,One Big Beautiful Bill Act,Yes,No,Yes,Yes
